# Clay-Organic-Ion-Water System Analysis

Comprehensive analysis of clay-organic-ion-water interactions using the ClayOrganicIonWaterAnalysis framework.

**Analysis Components:**
1. Multi-component radial distribution functions (RDFs)
2. Competitive adsorption analysis
3. Organic molecule conformations
4. Three-component bridge structures
5. Hydration shell competition
6. Stratified (layered) adsorption profiles
7. Exchange kinetics and residence times
8. Selectivity coefficients

**Author:** R.Swai  
**Date:** January 2026

In [ ]:
# Run this once at the beginning of your notebook
%load_ext autoreload
%autoreload 2

In [ ]:
# Core scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import pickle
import os
from datetime import datetime

# MDAnalysis for trajectory handling
import MDAnalysis as mda
from MDAnalysis.analysis.base import AnalysisBase, Results

# Progress bar
from tqdm import tqdm

# Import custom classes
import sys
sys.path.append('/Users/roev0007/Documents/solvation_shells/my_scripts')
from ClayOrganicIonWaterAnalysis import ClayOrganicIonWaterAnalysis
from ClayOrganicIonWaterAnalysisPlotter import ClayOrganicIonWaterAnalysisPlotter

print("✓ All imports successful")

## 1. Load Trajectory and Initialize Analysis

In [ ]:
# Check total frames available
u = mda.Universe('system.tpr', 'trajectory.trr')
total_frames = len(u.trajectory)
print(f"Total frames available: {total_frames}")
print(f"Simulation box dimensions: {u.dimensions}")

In [ ]:
# Initialize the analysis object
# Modify selections based on your system
analysis = ClayOrganicIonWaterAnalysis(
    trajectory_path='trajectory.trr',
    topology_path='system.tpr',
    
    # Clay surface atoms
    clay_selection='resname MMT',
    
    # Ion selections (modify based on your ions)
    ion_selections={
        'Na': 'name NA',
        'K': 'name K',
        'Mg': 'name MG',
        'Ca': 'name CA',
        'Cl': 'name CL'
    },
    
    # Organic molecule selections (modify for your system)
    organic_selections={
        'organic_1': 'resname ORG',
        'polymer': 'resname POL',
        # Add more organic species as needed
    },
    
    # Water selection
    water_selection='resname SOL or resname WAT'
)

print(f"\n✓ Analysis object initialized")
print(f"  Clay atoms: {len(analysis.clay)}")
print(f"  Water molecules: {len(analysis.water) // 3}")
print(f"  Ion types: {list(analysis.ions.keys())}")
print(f"  Organic types: {list(analysis.organics.keys())}")

## 2. Multi-Component RDF Analysis

Calculate radial distribution functions between all component pairs.

In [ ]:
# Calculate multi-component RDFs
# This analyzes all pairwise interactions: clay-ion, clay-organic, ion-water, etc.
rdf_results = analysis.calculate_multi_component_rdfs(
    components=None,  # None = analyze all components
    atom_types=None   # Can specify specific atom types if needed
)

print(f"\n✓ RDF calculations complete")
print(f"  Number of RDF pairs calculated: {len(rdf_results)}")
print(f"  RDF pairs:")
for pair_name in rdf_results.keys():
    print(f"    - {pair_name}")

In [ ]:
# Plot multi-component RDFs
plotter = ClayOrganicIonWaterAnalysisPlotter(analysis)

plotter.plot_multi_component_rdfs(
    pairs=None,  # Plot all pairs, or specify list like ['clay-Na', 'organic_1-water']
    show_coordination=True,  # Show coordination number subplots
    xlim=(0, 15),  # Distance range
    ylim=None,
    figsize=(18, 12),
    save_plots=True,
    filename='multi_component_rdfs.png',
    dpi=600
)

## 3. Competitive Adsorption Analysis

Analyze how ions and organics compete for binding sites on the clay surface.

In [ ]:
# Analyze competitive adsorption
# Tracks which species bind to clay surface and in what amounts
competitive_results = analysis.analyze_competitive_adsorption(
    distance_cutoff=3.5  # Angstroms - distance to consider as "bound"
)

print(f"\n✓ Competitive adsorption analysis complete")
print(f"\nIon surface contacts (mean ± std):")
for ion_name, data in competitive_results['ion_surface_contacts'].items():
    print(f"  {ion_name}: {data['mean']:.2f} ± {data['std']:.2f}")

print(f"\nOrganic surface contacts (mean ± std):")
for org_name, data in competitive_results['organic_surface_contacts'].items():
    print(f"  {org_name}: {data['mean']:.2f} ± {data['std']:.2f}")

In [ ]:
# Plot competitive adsorption results
plotter.plot_competitive_adsorption(
    show_time_series=True,  # Show dynamics over trajectory
    figsize=(14, 10),
    save_plots=True,
    filename='competitive_adsorption.png',
    dpi=600
)

## 4. Organic Molecule Conformation Analysis

Study shape and orientation of organic molecules in the system.

In [ ]:
# Analyze organic molecule conformations
# Calculates radius of gyration, aspect ratios, principal axes
conformation_results = analysis.analyze_organic_conformations()

print(f"\n✓ Organic conformation analysis complete")
for org_name, data in conformation_results.items():
    print(f"\n{org_name}:")
    print(f"  Radius of gyration: {data['radius_of_gyration']['mean']:.2f} ± {data['radius_of_gyration']['std']:.2f} Å")
    print(f"  Aspect ratio: {data['aspect_ratio']['mean']:.2f} ± {data['aspect_ratio']['std']:.2f}")

In [ ]:
# Plot organic conformations
plotter.plot_organic_conformations(
    organic_names=None,  # Plot all, or specify list
    figsize=(14, 10),
    save_plots=True,
    filename='organic_conformations.png',
    dpi=600
)

## 5. Three-Component Bridge Analysis

Identify bridging structures connecting three components (e.g., clay-ion-organic).

In [ ]:
# Analyze three-component bridges
# Identifies structures like clay-water-ion, ion-water-organic, etc.
bridge_results = analysis.analyze_three_component_bridges(
    bridge_cutoff=4.0  # Distance cutoff for bridge formation
)

print(f"\n✓ Three-component bridge analysis complete")
print(f"\nBridge types (mean ± std, max):")
for bridge_type, data in bridge_results.items():
    print(f"  {bridge_type}:")
    print(f"    Mean: {data['mean']:.2f} ± {data['std']:.2f}")
    print(f"    Max:  {data['max']:.0f}")

In [ ]:
# Plot three-component bridges
plotter.plot_three_component_bridges(
    show_time_series=True,  # Show dynamics
    figsize=(14, 10),
    save_plots=True,
    filename='three_component_bridges.png',
    dpi=600
)

## 6. Hydration Shell Competition

Analyze how ions and organics compete for water coordination.

In [ ]:
# Analyze hydration shell competition
# Studies water coordination around ions vs organics
hydration_results = analysis.analyze_hydration_shell_competition(
    shell_cutoffs=[3.5, 5.0]  # First and second shell cutoffs
)

print(f"\n✓ Hydration shell competition analysis complete")
for shell_name in ['shell_3.5A', 'shell_5.0A']:
    print(f"\n{shell_name}:")
    print(f"  Ion hydration numbers:")
    for ion_name, data in hydration_results['ion_hydration'][shell_name].items():
        print(f"    {ion_name}: {data['mean']:.2f} ± {data['std']:.2f}")
    
    print(f"  Organic hydration numbers:")
    for org_name, data in hydration_results['organic_hydration'][shell_name].items():
        print(f"    {org_name}: {data['mean']:.2f} ± {data['std']:.2f}")

In [ ]:
# Plot hydration shell competition
plotter.plot_hydration_shell_competition(
    shell_cutoffs=None,  # Plot all shells
    figsize=(14, 12),
    save_plots=True,
    filename='hydration_competition.png',
    dpi=600
)

## 7. Stratified Adsorption (Density Profiles)

Create density profiles perpendicular to the clay surface.

In [ ]:
# Analyze stratified adsorption
# Creates 1D density profiles along z-direction (perpendicular to surface)
stratified_results = analysis.analyze_stratified_adsorption(
    z_direction='z',  # Direction perpendicular to clay ('x', 'y', or 'z')
    bin_size=0.5      # Bin size in Angstroms
)

print(f"\n✓ Stratified adsorption analysis complete")
print(f"  Clay surface position: {stratified_results['clay_surface_position']:.2f} Å")
print(f"  Number of bins: {len(stratified_results['bin_centers'])}")
print(f"  Z-range: {stratified_results['bin_centers'].min():.2f} to {stratified_results['bin_centers'].max():.2f} Å")
print(f"  Components analyzed: {list(stratified_results['density_profiles'].keys())}")

In [ ]:
# Plot stratified adsorption profiles
plotter.plot_stratified_adsorption(
    components=None,  # Plot all components
    relative_position=True,  # Plot relative to clay surface (not absolute z)
    figsize=(12, 8),
    save_plots=True,
    filename='stratified_adsorption.png',
    dpi=600
)

## 8. Exchange Kinetics Analysis

Study adsorption-desorption dynamics and residence times.

In [ ]:
# Analyze exchange kinetics
# Calculates residence times and exchange rates for surface binding
kinetics_results = analysis.analyze_exchange_kinetics(
    distance_cutoff=4.0,
    time_step=None  # Auto-detect from trajectory
)

print(f"\n✓ Exchange kinetics analysis complete")

if kinetics_results['residence_times']:
    print(f"\nResidence times (mean ± std):")
    for species, data in kinetics_results['residence_times'].items():
        print(f"  {species}: {data['mean']:.2f} ± {data['std']:.2f} ps")
        print(f"    Number of events: {len(data['distribution'])}")

if kinetics_results['exchange_rates']:
    print(f"\nExchange rates (events/ps):")
    for species, rate in kinetics_results['exchange_rates'].items():
        print(f"  {species}: {rate:.4f}")

In [ ]:
# Plot exchange kinetics
plotter.plot_exchange_kinetics(
    species=None,  # Plot all species
    figsize=(14, 10),
    save_plots=True,
    filename='exchange_kinetics.png',
    dpi=600
)

## 9. Selectivity Coefficients

Calculate competitive binding preferences.

In [ ]:
# Calculate selectivity coefficients
# Quantifies binding preferences between different species
selectivity_results = analysis.calculate_selectivity_coefficients()

print(f"\n✓ Selectivity coefficient calculation complete")
print(f"\nSelectivity coefficients:")
for pair, coefficient in sorted(selectivity_results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {pair}: {coefficient:.3f}")

In [ ]:
# Plot selectivity coefficients
plotter.plot_selectivity_coefficients(
    log_scale=False,  # Use log scale if values span many orders of magnitude
    figsize=(12, 8),
    save_plots=True,
    filename='selectivity_coefficients.png',
    dpi=600
)

## 10. Comprehensive Summary

Generate overview dashboard and summary statistics.

In [ ]:
# Create comprehensive summary
summary = analysis.create_comprehensive_summary()

print("\n" + "="*70)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*70)

print("\n📊 System Composition:")
print(f"  Clay atoms: {summary['system_composition']['n_clay_atoms']}")
print(f"  Water molecules: {summary['system_composition']['n_water_molecules']}")
print(f"  Ions:")
for ion_name, count in summary['system_composition']['n_ions'].items():
    print(f"    {ion_name}: {count}")
print(f"  Organics:")
for org_name, count in summary['system_composition']['n_organics'].items():
    print(f"    {org_name}: {count}")
print(f"  Frames analyzed: {summary['system_composition']['n_frames']}")

print("\n✅ Completed Analyses:")
for analysis_name in summary['analysis_completed']:
    print(f"  - {analysis_name}")

print("\n🔑 Key Findings:")
if summary['key_findings']:
    for finding, value in summary['key_findings'].items():
        if isinstance(value, tuple):
            print(f"  {finding}: {value[0]} (value: {value[1]:.2f})")
        else:
            print(f"  {finding}: {value}")

print("\n" + "="*70)

In [ ]:
# Plot comprehensive summary dashboard
plotter.plot_comprehensive_summary(
    figsize=(18, 12),
    save_plots=True,
    filename='comprehensive_summary.png',
    dpi=600
)

## 11. Run Full Analysis (Alternative)

Optionally, run all analyses at once instead of individually.

In [ ]:
# Run all analyses in one go
# This executes all 8 analysis methods sequentially
full_results = analysis.run_full_analysis(
    save_results=True,
    output_file='clay_organic_ion_water_results.npz'
)

print("\n✓ Full analysis pipeline completed!")
print(f"  Results saved to: clay_organic_ion_water_results.npz")

## 12. Save and Load Results

Save computed results for later use without recomputing.

In [ ]:
# Manual save/load (if not using run_full_analysis)

# Save results manually
output_file = 'manual_save_results.npz'
np.savez_compressed(
    output_file,
    results=analysis.results,
    rdf_results=analysis.rdf_results,
    timestamp=datetime.now().isoformat()
)
print(f"✓ Results saved to: {output_file}")

In [ ]:
# Load previously saved results
loaded_data = np.load('clay_organic_ion_water_results.npz', allow_pickle=True)

# Access results
loaded_results = loaded_data['results'].item()
loaded_rdf = loaded_data['rdf_results'].item()
loaded_summary = loaded_data['summary'].item()

print("✓ Results loaded successfully")
print(f"  Available analyses: {list(loaded_results.keys())}")

## 13. Custom Analysis Examples

Additional examples for customizing the analysis workflow.

In [ ]:
# Example: Plot only specific RDF pairs
plotter.plot_multi_component_rdfs(
    pairs=['clay-Na', 'organic_1-water', 'Na-water'],
    show_coordination=False,
    xlim=(0, 10),
    figsize=(15, 5),
    save_plots=True,
    filename='selected_rdfs.png'
)

In [ ]:
# Example: Plot specific organic conformations
plotter.plot_organic_conformations(
    organic_names=['organic_1'],  # Only plot specific organics
    figsize=(14, 6),
    save_plots=True,
    filename='organic_1_conformation.png'
)

In [ ]:
# Example: Stratified adsorption with specific components
plotter.plot_stratified_adsorption(
    components=['water', 'Na', 'organic_1'],  # Only these components
    relative_position=True,
    figsize=(10, 6),
    save_plots=True,
    filename='selected_density_profiles.png'
)

## 14. Export Publication-Quality Figures

Tips for creating publication-ready figures.

In [ ]:
# Set custom matplotlib style for publications
import matplotlib as mpl

# Publication style settings
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['figure.titlesize'] = 18
mpl.rcParams['lines.linewidth'] = 2.5
mpl.rcParams['axes.linewidth'] = 1.5

print("✓ Publication style settings applied")

In [ ]:
# Re-plot with publication settings
plotter.plot_competitive_adsorption(
    show_time_series=False,  # Simpler for publication
    figsize=(8, 6),
    save_plots=True,
    filename='publication_competitive_adsorption.png',
    dpi=600
)

## Notes and Best Practices

**Performance Tips:**
- Use `step` parameter in analysis methods to skip frames for faster computation
- Save results with `run_full_analysis(save_results=True)` to avoid recomputation
- Check memory usage for large trajectories

**Customization:**
- Modify `distance_cutoff` values based on your system
- Adjust `bin_size` for stratified analysis depending on interface thickness
- Change `shell_cutoffs` for hydration analysis based on RDF peaks

**Troubleshooting:**
- Check atom selections with `len(analysis.clay)`, `len(analysis.water)`, etc.
- Verify trajectory loaded correctly with `print(u.trajectory)`
- Use smaller frame ranges for testing: `analysis.u.trajectory[0:100:10]`

**Citation:**
If using this workflow, please cite:
- MDAnalysis: doi.org/10.25080/Majora-629e541a-00e
- NumPy: doi.org/10.1038/s41586-020-2649-2
- Matplotlib: doi.org/10.1109/MCSE.2007.55